# Prism — Train HOG + SVM — Slot A (classical)
**Ron — HOG + SVM Lead**

Hand-written HOG descriptor + linear SVM (NumPy only, no sklearn) on CIFAR-10.

### Colab setup
1. Clone the repo: `!git clone https://github.com/JethroIsHere/prism.git`
   (or upload the project folder and adjust `PROJECT_ROOT` below).
2. Run the cells top to bottom. CPU is fine — the whole thing takes ~1 minute.


In [1]:
import json, os, sys, time
import numpy as np

# Locate the project root: this notebook lives in ron-hog-svm/
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'ron-hog-svm')):
    PROJECT_ROOT = _cwd
elif os.path.basename(_cwd) == 'ron-hog-svm':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = "/content/prism" if os.path.isdir("/content/prism") else "."
print("PROJECT_ROOT:", PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "ron-hog-svm"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "shared"))

from hog_svm import (hog_features, hog_dimension, LinearSVM,
                     StandardScaler, confusion_matrix,
                     accuracy_and_macro_f1)
from data_utils import load_cifar10_raw, CLASS_NAMES, SEED

RESULTS = os.path.join(PROJECT_ROOT, "ron-hog-svm", "results")
os.makedirs(RESULTS, exist_ok=True)

# 4x4 px cells suit 32x32 images (chosen on the 5k val split: 52.82% vs 48.08% for 8x8)
CELL_SIZE = 4
SVM_PARAMS = {"epochs": 25, "batch_size": 256, "lr": 0.1,
              "lr_decay": 0.92, "reg": 1e-3}
print("descriptor dim:", hog_dimension(cell_size=CELL_SIZE))


PROJECT_ROOT: /home/hatch/workspace/cv-final-project
descriptor dim: 1764


In [2]:
# CIFAR-10 download (one-time, ~170MB) — skipped if already present.
import tarfile, urllib.request
_DATA = os.path.join(PROJECT_ROOT, "data")
_READY = os.path.join(_DATA, "cifar-10-batches-py", "test_batch")
os.makedirs(_DATA, exist_ok=True)
if not os.path.isfile(_READY):
    print("Downloading CIFAR-10 (~170MB)...")
    _tgz = os.path.join(_DATA, "cifar-10-python.tar.gz")
    urllib.request.urlretrieve(
        "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", _tgz)
    print("Extracting...")
    with tarfile.open(_tgz, "r:gz") as _tf:
        for _m in _tf.getmembers():
            if _m.isfile():  # manual copy: avoids tar ownership quirks
                _out = os.path.join(_DATA, _m.name)
                os.makedirs(os.path.dirname(_out), exist_ok=True)
                with open(_out, 'wb') as _f:
                    _f.write(_tf.extractfile(_m).read())
    print("CIFAR-10 ready.")
else:
    print("CIFAR-10 already present.")


CIFAR-10 already present.


## 1. Sanity check

The core classical algorithm must be tested on synthetic images **before**
it touches the real dataset: blank → zeros, vertical/horizontal/diagonal
edges → the expected orientation bins. (Gradients are *perpendicular* to edges:
a y=x diagonal has gradients at 135°, i.e. bins 6–7.)


In [3]:
print('Running HOG sanity checks on synthetic images...')
blank = np.zeros((32, 32, 3))
vert = np.zeros((32, 32, 3)); vert[:, 16:, :] = 1.0
horiz = np.zeros((32, 32, 3)); horiz[16:, :, :] = 1.0
diag = np.zeros((32, 32, 3))
yy, xx = np.mgrid[0:32, 0:32]; diag[yy > xx] = 1.0
adiag = np.zeros((32, 32, 3)); adiag[yy + xx > 31] = 1.0
Xsyn = np.stack([blank, vert, horiz, diag, adiag])
D = hog_features(Xsyn)  # default 8px cells -> 324-dim (shape check only)
assert D.shape == (5, 324), D.shape
assert np.isfinite(D).all()
assert np.abs(D[0]).max() == 0.0, 'blank image must give zeros'
def bin_energy(d):
    return d.reshape(9, 4, 9).sum(axis=(0, 1))  # energy per orientation bin
checks = [("vertical edge -> bin 0", int(np.argmax(bin_energy(D[1]))) == 0),
          ("horizontal edge -> bin 4", int(np.argmax(bin_energy(D[2]))) == 4),
          ("diagonal edge -> bins 6-7", int(np.argmax(bin_energy(D[3]))) in (6, 7)),
          ("anti-diagonal -> bins 2-3", int(np.argmax(bin_energy(D[4]))) in (2, 3))]
for name, passed in checks:
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
assert all(p for _, p in checks), "sanity check failed — fix before training"
print("ALL SANITY CHECKS PASSED")


Running HOG sanity checks on synthetic images...
[PASS] vertical edge -> bin 0
[PASS] horizontal edge -> bin 4
[PASS] diagonal edge -> bins 6-7
[PASS] anti-diagonal -> bins 2-3
ALL SANITY CHECKS PASSED


## 2. Data

CIFAR-10: 50,000 train / 10,000 test images. (The 45k/5k train/val split
lives in `shared/data_utils.py` and was used to choose the 4px cell size;
the final model refits on the full 50k, standard practice.)


In [4]:
Xtr_u8, ytr, Xte_u8, yte = load_cifar10_raw(
    os.path.join(PROJECT_ROOT, 'data', 'cifar-10-batches-py'))
X_train = Xtr_u8.astype(np.float32) / 255.0
X_test = Xte_u8.astype(np.float32) / 255.0
print(f'train {X_train.shape}, test {X_test.shape}')
log = []


train (50000, 32, 32, 3), test (10000, 32, 32, 3)


## 3. HOG features

Hand-written descriptor: grayscale → gradients → 4px cell orientation
histograms (soft voting) → block L2-Hys normalization → **1764-dim** vector.
Computed in chunks to bound memory.


In [5]:
def hog_batched(images, batch_size=2048):
    parts = [hog_features(images[s:s+batch_size], cell_size=CELL_SIZE)
             for s in range(0, len(images), batch_size)]
    return np.concatenate(parts, axis=0)

t0 = time.perf_counter()
F_train = hog_batched(X_train)
t_hog_train = time.perf_counter() - t0
print(f"train features {F_train.shape} in {t_hog_train:.1f}s ({t_hog_train/len(X_train)*1000:.1f} ms/image)")


train features (50000, 1764) in 5.7s (0.1 ms/image)


## 4. Train the linear SVM

One-vs-rest, minibatch SGD on the hinge loss. Features are standardized
first (required — the hinge loss needs comparable scales).


In [6]:
scaler = StandardScaler()
F_train_s = scaler.fit_transform(F_train)
del F_train  # free memory
svm = LinearSVM(F_train_s.shape[1], 10, seed=SEED)
t0 = time.perf_counter()
svm.fit(F_train_s, ytr, verbose=True, **SVM_PARAMS)
t_svm = time.perf_counter() - t0
print(f"SVM training took {t_svm:.1f}s")
train_time_min = (t_hog_train + t_svm) / 60
log.append(f"training_time_minutes: {train_time_min:.1f}")


  epoch  1/25  lr=0.1000  loss=3.5190
  epoch  2/25  lr=0.0920  loss=2.2828
  epoch  3/25  lr=0.0846  loss=2.2148
  epoch  4/25  lr=0.0779  loss=2.1650
  epoch  5/25  lr=0.0716  loss=2.1259
  epoch  6/25  lr=0.0659  loss=2.0907
  epoch  7/25  lr=0.0606  loss=2.0587
  epoch  8/25  lr=0.0558  loss=2.0318
  epoch  9/25  lr=0.0513  loss=2.0032
  epoch 10/25  lr=0.0472  loss=1.9872
  epoch 11/25  lr=0.0434  loss=1.9657
  epoch 12/25  lr=0.0400  loss=1.9467
  epoch 13/25  lr=0.0368  loss=1.9328
  epoch 14/25  lr=0.0338  loss=1.9166
  epoch 15/25  lr=0.0311  loss=1.9003
  epoch 16/25  lr=0.0286  loss=1.8919
  epoch 17/25  lr=0.0263  loss=1.8778
  epoch 18/25  lr=0.0242  loss=1.8684
  epoch 19/25  lr=0.0223  loss=1.8586
  epoch 20/25  lr=0.0205  loss=1.8489
  epoch 21/25  lr=0.0189  loss=1.8405
  epoch 22/25  lr=0.0174  loss=1.8336
  epoch 23/25  lr=0.0160  loss=1.8268
  epoch 24/25  lr=0.0147  loss=1.8201
  epoch 25/25  lr=0.0135  loss=1.8147
SVM training took 43.6s


## 5. Evaluation (run once on the test set)


In [7]:
t0 = time.perf_counter()
F_test = hog_batched(X_test)
t_hog_test = time.perf_counter() - t0
F_test_s = scaler.transform(F_test)
t1 = time.perf_counter()
y_pred = svm.predict(F_test_s)
t_pred = time.perf_counter() - t1
ms_per_image = (t_hog_test + t_pred) / len(X_test) * 1000
print(f"{ms_per_image:.2f} ms/image (CPU)")

cm = confusion_matrix(yte, y_pred, 10)
acc, macro_f1, per_class_f1 = accuracy_and_macro_f1(cm)
print(f"Top-1 accuracy: {acc*100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
log.append(f"top1_accuracy_pct: {acc*100:.2f}")
log.append(f"macro_f1: {macro_f1:.4f}")

metrics = {
    "method": "HOG + linear SVM (from scratch, NumPy)",
    "dataset": "CIFAR-10",
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "descriptor_dim": int(F_test_s.shape[1]),
    "top1_accuracy_pct": round(acc*100, 2),
    "macro_f1": round(macro_f1, 4),
    "per_class_f1": {n: round(float(v), 4) for n, v in zip(CLASS_NAMES, per_class_f1)},
    "runtime_ms_per_image_cpu": round(ms_per_image, 2),
    "parameter_count_millions": 0,
    "learned_weights_actual": int(svm.W.size + svm.b.size),
    "training_time_minutes": round(train_time_min, 1),
    "hog_params": {"cell_size": CELL_SIZE, "n_bins": 9,
                 "block_cells": 2, "clip": 0.2},
    "svm_hyperparams": SVM_PARAMS,
}
with open(os.path.join(RESULTS, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS, "confusion_matrix.npy"), cm)
svm.save(os.path.join(RESULTS, "model.npz"))
np.savez(os.path.join(RESULTS, "scaler.npz"), mean=scaler.mean_, std=scaler.std_)
with open(os.path.join(RESULTS, "training_log.txt"), "w") as f:
    f.write("\n".join(log) + "\n")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'HOG + SVM confusion matrix (acc {acc*100:.2f}%)')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 'confusion_matrix.png'), dpi=150)
plt.close(fig)
print("Saved: metrics.json, confusion_matrix.png/.npy, model.npz, scaler.npz, training_log.txt")


0.14 ms/image (CPU)
Top-1 accuracy: 52.06%
Macro F1: 0.5116
Saved: metrics.json, confusion_matrix.png/.npy, model.npz, scaler.npz, training_log.txt
